Document Retrieval
to identify the product documents stored in ChromaDB that are most relevant to a user's question.

The retrieval process uses vector similarity search. The user's question is represented by an embedding, which is compared with the embeddings of the stored product documents.

The system then returns the most relevant documents, together with their metadata and source URLs.

Workflow:
The user submits a question.
Query preprocessing validates the question and generates its embedding.
The retriever searches the ChromaDB collection.
The most relevant product documents and their sourcesare returned.
The LLM uses the retrieved information to generate a grounded answer.

Retrieval Method
We use vector similarity search with ChromaDB.
The database stores product embeddings generated using paraphrase-multilingual-MiniLM-L12-v2.
The collection uses cosine distance as its distance metric.
The retrieved results are therefore ordered by their cosine distance, with smaller distances indicating greater similarity.

In [3]:
#Connecting to ChromaDB
import chromadb
import numpy as np



In [4]:
# Connect to the existing persistent database
client = chromadb.PersistentClient(path="./chroma_db")

# Retrieve the existing product collection
collection = client.get_collection(name="pharmaly_products")

print("Connected to ChromaDB successfully!")
print("Collection name:", collection.name)
print("Number of stored products:", collection.count())

Connected to ChromaDB successfully!
Collection name: pharmaly_products
Number of stored products: 9148


Understanding Vector Similarity Search
Cosine Similarity: The retrieval component uses vector similarity search to identify product documents related to the user's question.

Each product document and each user query is represented by a numerical vector.

Cosine similarity measures the cosine of the angle between two vectors
Cosine Distance in ChromaDB

Our ChromaDB collection was configured with:

hnsw:space = cosine

ChromaDB returns cosine distance rather than cosine similarity.
Ranking Retrieved Documents

ChromaDB returns the closest documents according to their distance.

We can convert the returned distances into similarity scores to make the results easier to interpret.

These scores help us evaluate retrieval quality, but they should not automatically be interpreted as probabilities or guarantees that a document answers the question.


In [5]:
#Performing a First Similarity Search
from embeddings import get_embedding

# Example user question
user_query = "Which products contain probiotics?"

# Generate the query embedding
query_embedding = get_embedding(user_query)

# Search for the 5 most similar product documents
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

print("Number of retrieved documents:", len(results["ids"][0]))

Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded!
Embedding dimensions: 384
Number of retrieved documents: 5


c:\Users\admin\Desktop\RAG-\rag\embeddings.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions:", model.get_sentence_embedding_dimension())


In [6]:
# Inspecting Retrieved Documents
# Display the retrieved products
for i in range(len(results["ids"][0])):

    document = results["documents"][0][i]
    metadata = results["metadatas"][0][i]
    distance = results["distances"][0][i]

    similarity = 1 - distance

    print(f"\n--- Result {i + 1} ---")
    print("Product:", metadata.get("product_name", "Unknown"))
    print("Brand:", metadata.get("brand", "Unknown"))
    print("Category:", metadata.get("category", "Unknown"))
    print("Cosine distance:", round(distance, 4))
    print("Cosine similarity:", round(similarity, 4))
    print("Product URL:", metadata.get("product_url", "Not available"))
    print("Document:", document[:300])


--- Result 1 ---
Product: Centella Probio Cica Essence Toner
Brand: SKIN1004
Category: K-Beauty, Visage
Cosine distance: 0.3147
Cosine similarity: 0.6853
Product URL: https://www.pharmaly-dz.com/produit/centella-probio-cica-essence-toner/
Document: 
Product: Centella Probio Cica Essence Toner
Brand: SKIN1004
Category: K-Beauty, Visage
Current price: 3500.0
Description: Ce tonique de la gamme avancée Probio-Cica est conçu pour les peaux qui ont besoin de plus qu’une simple hydratation. Il agit comme un véritable pansement liquide : Centella Asi

--- Result 2 ---
Product: Centella Probio Cica Crème Riche
Brand: SKIN1004
Category: K-Beauty, Visage
Cosine distance: 0.3391
Cosine similarity: 0.6609
Product URL: https://www.pharmaly-dz.com/produit/centella-probio-cica-creme-riche-50ml/
Document: 
Product: Centella Probio Cica Crème Riche
Brand: SKIN1004
Category: K-Beauty, Visage
Current price: 4200.0
Description: Cette crème est le fleuron de la gamme Probio-Cica. Elle s’adresse à celles e

In [7]:
#Build the retrieval function
def retrieve_documents(prepared_query, top_k=5):
    """
    Retrieve the most relevant product documents from ChromaDB.

    Parameters:
        prepared_query (dict): Cleaned query and its embedding.
        top_k (int): Maximum number of documents to retrieve.

    Returns:
        list: Retrieved documents with their scores and metadata.
    """

    # Extract the question and its embedding
    query = prepared_query["query"]
    query_embedding = prepared_query["embedding"]

    # Search the existing ChromaDB collection
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_documents = []

    # Organize the retrieved results
    for i in range(len(results["ids"][0])):

        distance = results["distances"][0][i]

        document_info = {
            "id": results["ids"][0][i],
            "query": query,
            "content": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": distance,
            "similarity": 1 - distance,
            "source_url": results["metadatas"][0][i].get("product_url")
        }

        retrieved_documents.append(document_info)

    return retrieved_documents


In [8]:
#Test the function
# Reuse the query and embedding from our previous test
prepared_query = {
    "query": user_query,
    "embedding": query_embedding
}

# Retrieve the top 5 documents
retrieved_documents = retrieve_documents(prepared_query, top_k=5)

print("Number of retrieved documents:", len(retrieved_documents))

for i, result in enumerate(retrieved_documents, start=1):
    print(f"\n--- Result {i} ---")
    print("Product:", result["metadata"].get("product_name"))
    print("Similarity:", round(result["similarity"], 4))
    print("Source:", result["source_url"])

Number of retrieved documents: 5

--- Result 1 ---
Product: Centella Probio Cica Essence Toner
Similarity: 0.6853
Source: https://www.pharmaly-dz.com/produit/centella-probio-cica-essence-toner/

--- Result 2 ---
Product: Centella Probio Cica Crème Riche
Similarity: 0.6609
Source: https://www.pharmaly-dz.com/produit/centella-probio-cica-creme-riche-50ml/

--- Result 3 ---
Product: Centella Probio Cica Intensive Ampoule 95ml
Similarity: 0.6589
Source: https://www.pharmaly-dz.com/produit/centella-probio-cica-intensive-ampoule-95ml/

--- Result 4 ---
Product: Vichy Minéral 89 Probiotic Fractions Concentré Protecteur 30ml
Similarity: 0.6428
Source: https://www.pharmaly-dz.com/produit/vichy-mineral-89-probiotic-fractions-concentre-protecteur-30ml/

--- Result 5 ---
Product: Biogaia Gouttes 5ml
Similarity: 0.6406
Source: https://www.pharmaly-dz.com/produit/biogaia-gouttes-5ml/


In [9]:
#Build the reusable retrieval function
def retrieve_documents(prepared_query, top_k=5):
    """
    Retrieve the most relevant documents from ChromaDB.

    Parameters:
        prepared_query: dictionary containing the query and its embedding.
        top_k: number of documents to retrieve.

    Returns:
        A list of retrieved documents with their metadata and scores.
    """

    # Extract the query and its embedding
    query = prepared_query["query"]
    query_embedding = prepared_query["embedding"]

    # Search the existing ChromaDB collection
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    # Organize the retrieved results
    retrieved_documents = []

    for i in range(len(results["ids"][0])):

        distance = results["distances"][0][i]

        document = {
            "id": results["ids"][0][i],
            "content": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": distance,
            "similarity": 1 - distance
        }

        retrieved_documents.append(document)

    return retrieved_documents

In [10]:
#Test the retrieval function
# Prepare a test query
test_query = {
    "query": "Which products contain probiotics?",
    "embedding": get_embedding("Which products contain probiotics?")
}

# Retrieve the most relevant documents
retrieved_documents = retrieve_documents(test_query, top_k=5)

# Display the results
print("Number of retrieved documents:", len(retrieved_documents))

for i, document in enumerate(retrieved_documents):

    print(f"\n--- Result {i + 1} ---")
    print("Product:", document["metadata"].get("product_name"))
    print("Similarity:", round(document["similarity"], 4))
    print("Source:", document["metadata"].get("product_url"))
    

Number of retrieved documents: 5

--- Result 1 ---
Product: Centella Probio Cica Essence Toner
Similarity: 0.6853
Source: https://www.pharmaly-dz.com/produit/centella-probio-cica-essence-toner/

--- Result 2 ---
Product: Centella Probio Cica Crème Riche
Similarity: 0.6609
Source: https://www.pharmaly-dz.com/produit/centella-probio-cica-creme-riche-50ml/

--- Result 3 ---
Product: Centella Probio Cica Intensive Ampoule 95ml
Similarity: 0.6589
Source: https://www.pharmaly-dz.com/produit/centella-probio-cica-intensive-ampoule-95ml/

--- Result 4 ---
Product: Vichy Minéral 89 Probiotic Fractions Concentré Protecteur 30ml
Similarity: 0.6428
Source: https://www.pharmaly-dz.com/produit/vichy-mineral-89-probiotic-fractions-concentre-protecteur-30ml/

--- Result 5 ---
Product: Biogaia Gouttes 5ml
Similarity: 0.6406
Source: https://www.pharmaly-dz.com/produit/biogaia-gouttes-5ml/


In [11]:
#Evaluate retrieval quality with different questions
# Define different test queries
test_queries = [
    "Do you have La Roche-Posay sunscreen?",
    "Which products contain vitamin C?",
    "What is the capital of Japan?"
]

# Evaluate retrieval for each query
for query in test_queries:

    print("\n" + "=" * 60)
    print("USER QUERY:", query)
    print("=" * 60)

    # Generate the query embedding
    prepared_query = {
        "query": query,
        "embedding": get_embedding(query)
    }

    # Retrieve the 5 most relevant documents
    retrieved_documents = retrieve_documents(prepared_query, top_k=5)

    # Display retrieved products
    for i, document in enumerate(retrieved_documents):

        metadata = document["metadata"]

        print(f"\nResult {i + 1}")
        print("Product:", metadata.get("product_name", "Unknown"))
        print("Similarity:", round(document["similarity"], 4))
        print("Source:", metadata.get("product_url", "Not available"))


USER QUERY: Do you have La Roche-Posay sunscreen?

Result 1
Product: La Roche-posay Anthelios 50+ Serum Sunscreen 50ml
Similarity: 0.6498
Source: https://www.pharmaly-dz.com/produit/lrp-anthelios-50-serum-sunscreen-50ml/

Result 2
Product: La Roche-posay Pigmentclar Uv Spf 30
Similarity: 0.6185
Source: https://www.pharmaly-dz.com/produit/lrp-pigmentclar-uv-spf-30-tube-40ml/

Result 3
Product: Garnier Ombrelle Face Sunscreen Crème Hydratante Spf60 75ml
Similarity: 0.6109
Source: https://www.pharmaly-dz.com/produit/garnier-ombrelle-face-sunscreen-creme-hydratante-spf60-75ml/

Result 4
Product: Yves Rocher Solaire Peau Parfaite Spf50+ 150ml
Similarity: 0.604
Source: https://www.pharmaly-dz.com/produit/yves-rocher-solaire-peau-parfaite-spf50-150ml/

Result 5
Product: Doumdoum Créme Écran Solaire Spf50 100ml
Similarity: 0.5982
Source: https://www.pharmaly-dz.com/produit/doumdoum-creme-ecran-solaire-spf50-100ml-2/

USER QUERY: Which products contain vitamin C?

Result 1
Product: Advanced Cl

In [12]:
# Inspect the retrieved content for the Vitamin C query

query = "Which products contain vitamin C?"

prepared_query = {
    "query": query,
    "embedding": get_embedding(query)
}

retrieved_documents = retrieve_documents(prepared_query, top_k=5)

for i, document in enumerate(retrieved_documents):

    print(f"\n{'=' * 60}")
    print(f"RESULT {i + 1}")
    print("=" * 60)

    print("Product:", document["metadata"].get("product_name"))
    print("Similarity:", round(document["similarity"], 4))

    print("\nProduct description:")
    print(document["content"])

    print("\nSource:")
    print(document["metadata"].get("product_url"))


RESULT 1
Product: Advanced Clinicals Brightening Vitamin C Scrub 237ml
Similarity: 0.7781

Product description:

Product: Advanced Clinicals Brightening Vitamin C Scrub 237ml
Brand: Advanced Clinicals
Category: Corps, Hygiene
Current price: 1300.0
Description: Le Gommage Éclat à la Vitamine C Advanced Clinicals est un gommage corps vitamine C à la texture crémeuse enrichie en particules exfoliantes. Il aide à éliminer les cellules mortes, lisser les zones rugueuses et raviver l’apparence des peaux ternes. Sa formule associe vitamine C, beurre de karité, huile de coco, thé vert et ginseng afin d’exfolier tout en préservant la douceur et le confort cutané. Son parfum d’agrumes apporte une sensation fraîche sous la douche. Ce gommage corps vitamine C convient particulièrement aux peaux sèches, rugueuses ou en manque d’éclat.


Source:
https://www.pharmaly-dz.com/produit/advanced-clinicals-brightening-vitamin-c-scrub-237ml/

RESULT 2
Product: The Ordinary Vitamine C
Similarity: 0.7161

Pr

In [13]:
# Inspect the metadata fields of one retrieved document

if retrieved_documents:
    metadata = retrieved_documents[0]["metadata"]

    print("Available metadata fields:")

    for key, value in metadata.items():
        print(f"{key}: {value}")
else:
    print("No documents were retrieved.")

Available metadata fields:
product_url: https://www.pharmaly-dz.com/produit/advanced-clinicals-brightening-vitamin-c-scrub-237ml/
current_price: 1300.0
category: Corps, Hygiene
brand: Advanced Clinicals
product_name: Advanced Clinicals Brightening Vitamin C Scrub 237ml


In [14]:
# Verify the structure returned by the retrieval function

test_query = {
    "query": "Which products contain vitamin C?",
    "embedding": get_embedding("Which products contain vitamin C?")
}

retrieved_documents = retrieve_documents(test_query, top_k=5)

# Check the output structure
assert isinstance(retrieved_documents, list)

for document in retrieved_documents:

    assert "id" in document
    assert "content" in document
    assert "metadata" in document
    assert "distance" in document
    assert "similarity" in document

    assert document["content"]
    assert document["metadata"].get("product_url")

print("All retrieval output checks passed!")
print("Number of retrieved documents:", len(retrieved_documents))
print("Available fields:", list(retrieved_documents[0].keys()))

All retrieval output checks passed!
Number of retrieved documents: 5
Available fields: ['id', 'content', 'metadata', 'distance', 'similarity']


In [15]:
#Retrieval Performance Evaluation
import time
import statistics

# Prepare the query embedding once
performance_query = {
    "query": "Which products contain vitamin C?",
    "embedding": get_embedding("Which products contain vitamin C?")
}

# Warm-up run (not included in the measurements)
retrieve_documents(performance_query, top_k=5)

# Measure retrieval time over 20 runs
retrieval_times = []

for _ in range(20):

    start_time = time.perf_counter()

    retrieved_documents = retrieve_documents(
        performance_query,
        top_k=5
    )

    end_time = time.perf_counter()

    retrieval_times.append(end_time - start_time)

# Calculate performance statistics
average_time = statistics.mean(retrieval_times)
minimum_time = min(retrieval_times)
maximum_time = max(retrieval_times)

print("RETRIEVAL PERFORMANCE")
print("-" * 40)
print("Number of runs:", len(retrieval_times))
print("Documents retrieved:", len(retrieved_documents))
print("Average time:", round(average_time * 1000, 2), "ms")
print("Minimum time:", round(minimum_time * 1000, 2), "ms")
print("Maximum time:", round(maximum_time * 1000, 2), "ms")

RETRIEVAL PERFORMANCE
----------------------------------------
Number of runs: 20
Documents retrieved: 5
Average time: 6.04 ms
Minimum time: 4.11 ms
Maximum time: 11.8 ms


In [16]:
#Retrieval Relevance Benchmark
# Define the benchmark queries
benchmark_queries = [
    "Do you have La Roche-Posay sunscreen?",
    "Which products contain vitamin C?",
    "Show me moisturizers for the face.",
    "Which products are suitable for acne-prone skin?",
    "Do you have The Ordinary Niacinamide 10%?",
    "What is the capital of Japan?"
]

# Retrieve and display results for each query
for query in benchmark_queries:

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5,
        include=["documents", "metadatas", "distances"]
    )

    for i in range(len(results["ids"][0])):

        metadata = results["metadatas"][0][i]
        distance = results["distances"][0][i]

        print(f"\nRank: {i + 1}")
        print("Product:", metadata.get("product_name"))
        print("Brand:", metadata.get("brand"))
        print("Category:", metadata.get("category"))
        print("Similarity:", round(1 - distance, 4))
        print("Source:", metadata.get("product_url"))


QUERY: Do you have La Roche-Posay sunscreen?

Rank: 1
Product: La Roche-posay Anthelios 50+ Serum Sunscreen 50ml
Brand: La Roche-Posay
Category: Solaire
Similarity: 0.6498
Source: https://www.pharmaly-dz.com/produit/lrp-anthelios-50-serum-sunscreen-50ml/

Rank: 2
Product: La Roche-posay Pigmentclar Uv Spf 30
Brand: La Roche-Posay
Category: Visage
Similarity: 0.6185
Source: https://www.pharmaly-dz.com/produit/lrp-pigmentclar-uv-spf-30-tube-40ml/

Rank: 3
Product: Garnier Ombrelle Face Sunscreen Crème Hydratante Spf60 75ml
Brand: Garnier
Category: Solaire
Similarity: 0.6109
Source: https://www.pharmaly-dz.com/produit/garnier-ombrelle-face-sunscreen-creme-hydratante-spf60-75ml/

Rank: 4
Product: Yves Rocher Solaire Peau Parfaite Spf50+ 150ml
Brand: Yves Rocher
Category: Solaire
Similarity: 0.604
Source: https://www.pharmaly-dz.com/produit/yves-rocher-solaire-peau-parfaite-spf50-150ml/

Rank: 5
Product: Doumdoum Créme Écran Solaire Spf50 100ml
Brand: Doumdoum
Category: Solaire
Similarity:

In [17]:
#Calculating Retrieval Metrics
# Relevance judgments for the six benchmark queries
# Each list corresponds to ranks 1 through 5.

relevance_labels = {
    "La Roche-Posay sunscreen": [1, 1, 0, 0, 0],
    "Vitamin C": [1, 1, 1, 1, 1],
    "Facial moisturizers": [0, 0, 0, 0, 1],
    "Acne-prone skin": [1, 1, 1, 0, 1],
    "The Ordinary Niacinamide 10%": [1, 0, 1, 1, 0],
    "Capital of Japan": [0, 0, 0, 0, 0]
}

precision_scores = []
reciprocal_ranks = []
hits = []

for query, labels in relevance_labels.items():

    # Precision@5
    precision = sum(labels) / len(labels)

    # Reciprocal rank of the first relevant result
    reciprocal_rank = 0

    for rank, label in enumerate(labels, start=1):
        if label == 1:
            reciprocal_rank = 1 / rank
            break

    # Hit: at least one relevant result
    hit = 1 if sum(labels) > 0 else 0

    precision_scores.append(precision)
    reciprocal_ranks.append(reciprocal_rank)
    hits.append(hit)

    print("\nQuery:", query)
    print("Precision@5:", round(precision, 3))
    print("Reciprocal Rank:", round(reciprocal_rank, 3))
    print("Hit:", hit)

# Overall metrics
print("\n" + "=" * 50)
print("OVERALL RETRIEVAL PERFORMANCE")
print("=" * 50)

print("Mean Precision@5:", round(np.mean(precision_scores), 3))
print("Hit Rate@5:", round(np.mean(hits), 3))
print("MRR@5:", round(np.mean(reciprocal_ranks), 3))


Query: La Roche-Posay sunscreen
Precision@5: 0.4
Reciprocal Rank: 1.0
Hit: 1

Query: Vitamin C
Precision@5: 1.0
Reciprocal Rank: 1.0
Hit: 1

Query: Facial moisturizers
Precision@5: 0.2
Reciprocal Rank: 0.2
Hit: 1

Query: Acne-prone skin
Precision@5: 0.8
Reciprocal Rank: 1.0
Hit: 1

Query: The Ordinary Niacinamide 10%
Precision@5: 0.6
Reciprocal Rank: 1.0
Hit: 1

Query: Capital of Japan
Precision@5: 0.0
Reciprocal Rank: 0
Hit: 0

OVERALL RETRIEVAL PERFORMANCE
Mean Precision@5: 0.5
Hit Rate@5: 0.833
MRR@5: 0.7


In [18]:
# Inspect potentially relevant and irrelevant results

queries_to_inspect = [
    "Show me moisturizers for the face.",
    "Which products are suitable for acne-prone skin?"
]

for query in queries_to_inspect:

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5,
        include=["documents", "metadatas", "distances"]
    )

    for i in range(len(results["ids"][0])):

        metadata = results["metadatas"][0][i]

        print(f"\n--- Rank {i + 1} ---")
        print("Product:", metadata.get("product_name"))
        print("Category:", metadata.get("category"))
        print("Description:")
        print(results["documents"][0][i])


QUERY: Show me moisturizers for the face.

--- Rank 1 ---
Product: Dr Melaxin Melting Cleanser Bubble Foaming Cleanser 150ml
Category: K-Beauty, Visage
Description:

Product: Dr Melaxin Melting Cleanser Bubble Foaming Cleanser 150ml
Brand: Dr. Melaxin
Category: K-Beauty, Visage
Current price: 4200.0
Description: Le nettoyant visage Dr Melaxin se distingue par sa mousse fine qui se transforme progressivement au contact de la peau pour dissoudre les impuretés, l’écran solaire et le maquillage léger. Ses bulles micellaires permettent un nettoyage efficace avec peu de friction, tandis que la glycérine aide à préserver le confort après le rinçage. Enrichi en niacinamide, calamine, aloe vera, thé vert et centella asiatica, le nettoyant visage Dr Melaxin convient à tous les types de peau, y compris sensibles. Utilisé sur peau sèche pour le démaquillage ou sur peau humide au quotidien, il laisse le visage propre, frais et souple.


--- Rank 2 ---
Product: Dermactive Fleure De Lys Masque Feuil

In [20]:
relevance_labels = {
    "La Roche-Posay sunscreen": [1, 1, 0, 0, 0],
    "Vitamin C": [1, 1, 1, 1, 1],
    "Facial moisturizers": [0, 1, 0, 1, 1],
    "Acne-prone skin": [1, 1, 1, 1, 1],
    "The Ordinary Niacinamide 10%": [1, 0, 1, 1, 0],
    "Capital of Japan": [0, 0, 0, 0, 0]
}
# Calculating Retrieval Metrics Using Corrected Labels

precision_scores = []
reciprocal_ranks = []
hits = []

for query, labels in relevance_labels.items():

    # Calculate Precision@5
    precision = sum(labels) / len(labels)

    # Calculate Reciprocal Rank
    reciprocal_rank = 0

    for rank, label in enumerate(labels, start=1):
        if label == 1:
            reciprocal_rank = 1 / rank
            break

    # Calculate Hit Rate
    hit = 1 if sum(labels) > 0 else 0

    precision_scores.append(precision)
    reciprocal_ranks.append(reciprocal_rank)
    hits.append(hit)

# Display overall metrics
print("OVERALL RETRIEVAL PERFORMANCE")
print("=" * 45)

print("Mean Precision@5:", round(np.mean(precision_scores), 3))
print("Hit Rate@5:", round(np.mean(hits), 3))
print("MRR@5:", round(np.mean(reciprocal_ranks), 3))

OVERALL RETRIEVAL PERFORMANCE
Mean Precision@5: 0.6
Hit Rate@5: 0.833
MRR@5: 0.75


In [22]:
print("Benchmark-related variables:")

for name in ["relevance_labels", "benchmark_results", "all_results",
             "test_queries", "retrieval_results"]:
    print(name, "exists:", name in globals())

Benchmark-related variables:
relevance_labels exists: True
benchmark_results exists: False
all_results exists: False
test_queries exists: True
retrieval_results exists: False


In [24]:
print("Type:", type(test_queries))
print("Contents:")
print(test_queries)


Type: <class 'list'>
Contents:
['Do you have La Roche-Posay sunscreen?', 'Which products contain vitamin C?', 'What is the capital of Japan?']


In [25]:
print("Available variables containing 'query' or 'result':")

for name in globals():
    if "query" in name.lower() or "result" in name.lower():
        value = globals()[name]
        if isinstance(value, (list, dict)):
            print(f"\n{name}: {type(value).__name__}, length={len(value)}")

Available variables containing 'query' or 'result':

query_embedding: list, length=384

results: dict, length=8

prepared_query: dict, length=2

result: dict, length=7

test_query: dict, length=2

performance_query: dict, length=2

query_max_scores: dict, length=0


In [26]:
print("Results keys:", results.keys())

for key, value in results.items():
    print(f"\n{key}:")
    print("Type:", type(value))

    if isinstance(value, list):
        print("Length:", len(value))
        if value:
            print("First item:", value[0])

Results keys: dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

ids:
Type: <class 'list'>
Length: 1
First item: ['8283', '7277', '4362', '6725', '1031']

embeddings:
Type: <class 'NoneType'>

documents:
Type: <class 'list'>
Length: 1
First item: ['\nProduct: Panoxyl Clarying Exfoliant For Acne Pore Skin 2% Salicylic 118ml\nBrand: PanOxyl\nCategory: Visage\nCurrent price: 3900.0\nDescription: L’ exfoliant anti-acné PanOxyl à 2% d’ acide salicylique est spécialement formulé pour les peaux à tendance acnéique et sujettes aux pores obstrués. Grâce à son action kératolytique, il exfolie en douceur, élimine les cellules mortes et désobstrue les pores pour prévenir l’apparition des imperfections. Sa formule aide à réduire l’excès de sébum tout en affinant le grain de peau. Utilisé régulièrement, il améliore visiblement la clarté du teint et limite les points noirs. Adapté aux routines ciblées, cet exfoliant favorise une peau plus nette , plus

In [27]:

# Reuse the six existing benchmark queries and relevance labels

benchmark_queries = {
    "La Roche-Posay sunscreen": "Do you have La Roche-Posay sunscreen?",
    "Vitamin C": "Which products contain vitamin C?",
    "Facial moisturizers": "Show me moisturizers for the face.",
    "Acne-prone skin": "Which products are suitable for acne-prone skin?",
    "The Ordinary Niacinamide 10%": "Do you have The Ordinary Niacinamide 10%?",
    "Capital of Japan": "What is the capital of Japan?"
}

# Retrieve documents and collect the maximum similarity score
threshold_benchmark = {}

for label, query in benchmark_queries.items():

    prepared = {
        "query": query,
        "embedding": get_embedding(query)
    }

    documents = retrieve_documents(prepared, top_k=5)

    max_similarity = max(
        doc["similarity"] for doc in documents
    )

    # A query is relevant if at least one retrieved document
    # was manually judged relevant in our benchmark.
    has_relevant_result = any(relevance_labels[label])

    threshold_benchmark[label] = {
        "query": query,
        "max_similarity": max_similarity,
        "has_relevant_result": has_relevant_result
    }

# Display collected scores
print("THRESHOLD EVALUATION DATA")
print("=" * 65)

for label, data in threshold_benchmark.items():
    print(f"\n{label}")
    print("Maximum similarity:", round(data["max_similarity"], 4))
    print("Has relevant result:", data["has_relevant_result"])


THRESHOLD EVALUATION DATA

La Roche-Posay sunscreen
Maximum similarity: 0.6498
Has relevant result: True

Vitamin C
Maximum similarity: 0.7781
Has relevant result: True

Facial moisturizers
Maximum similarity: 0.6535
Has relevant result: True

Acne-prone skin
Maximum similarity: 0.7504
Has relevant result: True

The Ordinary Niacinamide 10%
Maximum similarity: 0.5889
Has relevant result: True

Capital of Japan
Maximum similarity: 0.297
Has relevant result: False


In [28]:

# Compare Candidate Retrieval Thresholds

thresholds = [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

print("THRESHOLD EVALUATION")
print("=" * 85)

for threshold in thresholds:

    true_acceptances = 0
    false_acceptances = 0
    true_rejections = 0
    false_rejections = 0

    for data in threshold_benchmark.values():

        # Predict whether the query has sufficient retrieval evidence
        predicted_acceptance = data["max_similarity"] >= threshold

        # Compare prediction with our manual relevance judgment
        actual_relevance = data["has_relevant_result"]

        if predicted_acceptance and actual_relevance:
            true_acceptances += 1

        elif predicted_acceptance and not actual_relevance:
            false_acceptances += 1

        elif not predicted_acceptance and not actual_relevance:
            true_rejections += 1

        else:
            false_rejections += 1

    accuracy = (
        true_acceptances + true_rejections
    ) / len(threshold_benchmark)

    print(f"\nThreshold: {threshold:.2f}")
    print(f"True acceptances:  {true_acceptances}")
    print(f"False acceptances: {false_acceptances}")
    print(f"True rejections:   {true_rejections}")
    print(f"False rejections:  {false_rejections}")
    print(f"Accuracy:          {accuracy:.2%}")


THRESHOLD EVALUATION

Threshold: 0.30
True acceptances:  5
False acceptances: 0
True rejections:   1
False rejections:  0
Accuracy:          100.00%

Threshold: 0.40
True acceptances:  5
False acceptances: 0
True rejections:   1
False rejections:  0
Accuracy:          100.00%

Threshold: 0.45
True acceptances:  5
False acceptances: 0
True rejections:   1
False rejections:  0
Accuracy:          100.00%

Threshold: 0.50
True acceptances:  5
False acceptances: 0
True rejections:   1
False rejections:  0
Accuracy:          100.00%

Threshold: 0.55
True acceptances:  5
False acceptances: 0
True rejections:   1
False rejections:  0
Accuracy:          100.00%

Threshold: 0.60
True acceptances:  4
False acceptances: 0
True rejections:   1
False rejections:  1
Accuracy:          83.33%

Threshold: 0.65
True acceptances:  3
False acceptances: 0
True rejections:   1
False rejections:  2
Accuracy:          66.67%

Threshold: 0.70
True acceptances:  2
False acceptances: 0
True rejections:   1
False

In [29]:

# Retrieval Threshold Implementation

DEFAULT_SIMILARITY_THRESHOLD = 0.50

def apply_retrieval_threshold(
    retrieved_documents,
    threshold=DEFAULT_SIMILARITY_THRESHOLD
):
    """
    Filter retrieved documents using a minimum similarity score.

    Returns:
        A dictionary containing the retrieval status,
        accepted documents, maximum similarity and threshold.
    """

    if not 0 <= threshold <= 1:
        raise ValueError("Threshold must be between 0 and 1.")

    if not retrieved_documents:
        return {
            "status": "insufficient_evidence",
            "documents": [],
            "max_similarity": None,
            "threshold": threshold
        }

    # Keep only documents that meet the threshold
    accepted_documents = [
        document
        for document in retrieved_documents
        if document["similarity"] >= threshold
    ]

    # Calculate the highest similarity score
    max_similarity = max(
        document["similarity"]
        for document in retrieved_documents
    )

    # Determine whether sufficient evidence was retrieved
    status = (
        "sufficient_evidence"
        if accepted_documents
        else "insufficient_evidence"
    )

    return {
        "status": status,
        "documents": accepted_documents,
        "max_similarity": max_similarity,
        "threshold": threshold
    }


In [30]:

# Test 1: Relevant question
query = "Which products contain vitamin C?"

prepared_query = {
    "query": query,
    "embedding": get_embedding(query)
}

retrieved_documents = retrieve_documents(prepared_query, top_k=5)

threshold_result = apply_retrieval_threshold(retrieved_documents)

print("QUERY:", query)
print("STATUS:", threshold_result["status"])
print("MAXIMUM SIMILARITY:", round(threshold_result["max_similarity"], 4))
print("ACCEPTED DOCUMENTS:", len(threshold_result["documents"]))

# Test 2: Out-of-domain question
query = "What is the capital of Japan?"

prepared_query = {
    "query": query,
    "embedding": get_embedding(query)
}

retrieved_documents = retrieve_documents(prepared_query, top_k=5)

threshold_result = apply_retrieval_threshold(retrieved_documents)

print("\nQUERY:", query)
print("STATUS:", threshold_result["status"])
print("MAXIMUM SIMILARITY:", round(threshold_result["max_similarity"], 4))
print("ACCEPTED DOCUMENTS:", len(threshold_result["documents"]))


QUERY: Which products contain vitamin C?
STATUS: sufficient_evidence
MAXIMUM SIMILARITY: 0.7781
ACCEPTED DOCUMENTS: 5

QUERY: What is the capital of Japan?
STATUS: insufficient_evidence
MAXIMUM SIMILARITY: 0.297
ACCEPTED DOCUMENTS: 0


In [32]:
print("Status:", threshold_result["status"])
print("Number of accepted documents:", len(threshold_result["documents"]))
print("Threshold:", threshold_result["threshold"])

Status: insufficient_evidence
Number of accepted documents: 0
Threshold: 0.5


In [33]:
query = "Which products contain vitamin C?"

prepared_query = {
    "query": query,
    "embedding": get_embedding(query)
}

retrieved_documents = retrieve_documents(prepared_query, top_k=5)

threshold_result = apply_retrieval_threshold(retrieved_documents)

print("Status:", threshold_result["status"])
print("Accepted documents:", len(threshold_result["documents"]))

Status: sufficient_evidence
Accepted documents: 5


In [34]:

# Verify source information for accepted documents

required_fields = [
    "product_name",
    "brand",
    "product_url"
]

print("SOURCE METADATA VALIDATION")
print("=" * 50)

for i, document in enumerate(threshold_result["documents"], start=1):

    metadata = document["metadata"]

    print(f"\nDocument {i}: {metadata.get('product_name')}")

    for field in required_fields:
        value = metadata.get(field)

        print(f"{field}: {'OK' if value else 'MISSING'}")

    print("Similarity:", round(document["similarity"], 4))
    print("Content available:", bool(document["content"]))


SOURCE METADATA VALIDATION

Document 1: Advanced Clinicals Brightening Vitamin C Scrub 237ml
product_name: OK
brand: OK
product_url: OK
Similarity: 0.7781
Content available: True

Document 2: The Ordinary Vitamine C
product_name: OK
brand: OK
product_url: OK
Similarity: 0.7161
Content available: True

Document 3: Advanced Clinicals Vitamin C Sérum 52ml
product_name: OK
brand: OK
product_url: OK
Similarity: 0.7099
Content available: True

Document 4: Byphasse Vitamine C Crème Illuminatrice 50ml
product_name: OK
brand: OK
product_url: OK
Similarity: 0.701
Content available: True

Document 5: Avène Vitamin Active Cg Sérum Correcteure Éclat 30ml
product_name: OK
brand: OK
product_url: OK
Similarity: 0.6922
Content available: True


In [35]:
import os
print(os.getcwd())

c:\Users\admin\Desktop\RAG-\rag


In [36]:
import os

print("Working directory:", os.getcwd())
print("retrieval.py already exists:", os.path.exists("retrieval.py"))

Working directory: c:\Users\admin\Desktop\RAG-\rag
retrieval.py already exists: False


In [38]:

# Final Retrieval Threshold Validation

print("FINAL THRESHOLD VALIDATION")
print("=" * 65)

for label, query in benchmark_queries.items():

    prepared = {
        "query": query,
        "embedding": get_embedding(query)
    }

    retrieved_documents = retrieve_documents(
        prepared,
        top_k=5
    )

    threshold_result = apply_retrieval_threshold(
        retrieved_documents,
        threshold=DEFAULT_SIMILARITY_THRESHOLD
    )

    expected_relevance = any(relevance_labels[label])

    actual_acceptance = (
        threshold_result["status"] == "sufficient_evidence"
    )

    passed = expected_relevance == actual_acceptance

    print(f"\n{label}")
    print("Maximum similarity:",
          round(threshold_result["max_similarity"], 4))
    print("Expected:", "Accept" if expected_relevance else "Reject")
    print("Actual:", "Accept" if actual_acceptance else "Reject")
    print("Test:", "PASS" if passed else "FAIL")

print("\nFinal validation completed.")


FINAL THRESHOLD VALIDATION

La Roche-Posay sunscreen
Maximum similarity: 0.6498
Expected: Accept
Actual: Accept
Test: PASS

Vitamin C
Maximum similarity: 0.7781
Expected: Accept
Actual: Accept
Test: PASS

Facial moisturizers
Maximum similarity: 0.6535
Expected: Accept
Actual: Accept
Test: PASS

Acne-prone skin
Maximum similarity: 0.7504
Expected: Accept
Actual: Accept
Test: PASS

The Ordinary Niacinamide 10%
Maximum similarity: 0.5889
Expected: Accept
Actual: Accept
Test: PASS

Capital of Japan
Maximum similarity: 0.297
Expected: Reject
Actual: Reject
Test: PASS

Final validation completed.
